In [2]:
import logging
import os
from pathlib import Path
from abc import ABC, abstractmethod
from typing import Literal

from llama_index.readers.file import PDFReader
from llama_parse import LlamaParse
from app.core.exceptions import DocumentParsingError

logger = logging.getLogger("app_logger")

# --- 1. The Strategy Interface ---
class ParsingStrategy(ABC):
    @abstractmethod
    def parse(self, file_path: Path) -> str:
        """Parses the document and returns clean Markdown/Text."""
        pass

# --- 2. Concrete Strategies (The 4 Tiers) ---

class LlamaIndexLiteStrategy(ParsingStrategy):
    """Tier 1: LiteParse (Zero Cost) - Uses local LlamaIndex PDFReader."""
    def parse(self, file_path: Path) -> str:
        logger.info(f"Executing Tier 1: LlamaIndex LiteParse (Local/Zero Cost) on {file_path.name}")
        try:
            reader = PDFReader()
            documents = reader.load_data(file_path=file_path)
            # LlamaIndex local readers return Document objects; we join their text
            return "\n\n".join([doc.text for doc in documents])
        except Exception as e:
            logger.error(f"LiteParse failed: {str(e)}")
            raise DocumentParsingError(f"LiteParse failed: {str(e)}") from e

class LlamaParseFastStrategy(ParsingStrategy):
    """Tier 2: Fast (1 Credit) - Basic LlamaCloud parsing."""
    def parse(self, file_path: Path) -> str:
        logger.info(f"Executing Tier 2: LlamaParse Fast (1 Credit) on {file_path.name}")
        return self._run_llama_parse(file_path, premium_mode=False)

    def _run_llama_parse(self, file_path: Path, premium_mode: bool, custom_prompt: str = None) -> str:
        try:
            parser = LlamaParse(
                api_key=os.getenv("LLAMA_CLOUD_API_KEY"),
                result_type="markdown",
                premium_mode=premium_mode,
                parsing_instruction=custom_prompt
            )
            documents = parser.load_data(str(file_path))
            return "\n\n".join([doc.text for doc in documents])
        except Exception as e:
            logger.error(f"LlamaParse API failed: {str(e)}")
            raise DocumentParsingError(f"LlamaParse API failed: {str(e)}") from e

class LlamaParseCostEffectiveStrategy(LlamaParseFastStrategy):
    """Tier 3: Cost Effective (3 Credits) - Fast mode + specialized layout prompting."""
    def parse(self, file_path: Path) -> str:
        logger.info(f"Executing Tier 3: LlamaParse Cost-Effective (3 Credits) on {file_path.name}")
        # We use fast mode but add instructions to preserve tables/structure, 
        # which uses a slightly heavier backend process but avoids full agentic mode.
        prompt = "Strictly preserve markdown tables, headings, and lists. Do not hallucinate."
        return self._run_llama_parse(file_path, premium_mode=False, custom_prompt=prompt)

class LlamaParseAgenticStrategy(LlamaParseFastStrategy):
    """Tier 4: Agentic (10+ Credits) - Full premium VLM processing for equations/complex diagrams."""
    def parse(self, file_path: Path) -> str:
        logger.info(f"Executing Tier 4: LlamaParse Agentic (10+ Credits) on {file_path.name}")
        return self._run_llama_parse(file_path, premium_mode=True)


# --- 3. The Router (Context) ---
class DocumentRouter:
    """Routes documents to the appropriate Llama ecosystem parsing strategy."""
    def __init__(self):
        self.strategies = {
            "lite": LlamaIndexLiteStrategy(),
            "fast": LlamaParseFastStrategy(),
            "cost_effective": LlamaParseCostEffectiveStrategy(),
            "agentic": LlamaParseAgenticStrategy()
        }

    def _detect_complexity_heuristics(self, file_path: Path) -> Literal["high", "medium", "low"]:
        """Scans the document size locally to make a quick routing decision if the user is unsure."""
        # Since we want to drop PyMuPDF, we can use file size as a proxy heuristic for complexity
        size_kb = os.path.getsize(file_path) / 1024
        if size_kb > 5000: return "high"     # > 5MB usually means lots of images/diagrams
        if size_kb > 1000: return "medium"   # 1MB - 5MB 
        return "low"

    def route_and_parse(self, file_path: Path, user_category: str) -> str:
        if not file_path.exists():
            raise FileNotFoundError(f"Uploaded file missing at {file_path}")

        selected_tier = "fast" 
        
        if user_category == "Mostly Text":
            selected_tier = "lite"
        elif user_category == "Text with Tables":
            selected_tier = "cost_effective"
        elif user_category in ["Text with Diagrams/Figures", "Text with Equations", "Scanned PDF"]:
            selected_tier = "agentic"
        elif user_category == "I'm Not Sure":
            logger.info("User unsure. Running local heuristic scan...")
            complexity = self._detect_complexity_heuristics(file_path)
            if complexity == "high": selected_tier = "agentic"
            elif complexity == "medium": selected_tier = "cost_effective"
            else: selected_tier = "fast"
                
        strategy = self.strategies.get(selected_tier, self.strategies["fast"])
        return strategy.parse(file_path)

ModuleNotFoundError: No module named 'llama_index'

In [1]:
from liteparse import LiteParse

parser = LiteParse()
result = parser.parse("samples/c10-science-ch10-eng.pdf")

In [2]:
from rich import print as p

In [3]:
print(result.text)

     CHAPTER10
     The Human Eye and
     the Colourful World

    Y          ou have studied in the previous chapter about refraction of light by
                 lenses. You also studied the nature, position and relative size of
    images formed by lenses. How can these ideas help us in the study of
    the human eye? The human eye uses light and enables us to see objects
    around us. It has a lens in its structure. What is the function of the lens
    in a human eye? How do the lenses used in spectacles correct defects of
    vision? Let us consider these questions in this chapter.
                 We have learnt in the previous chapter about light and some of its
    properties. In this chapter, we shall use these ideas to study some of the
    optical phenomena in nature. We shall also discuss about rainbow
    formation, splitting of white light and blue colour of the sky.

    10.1 THE HUMAN EYE
    The human eye is one of the most valuable and sensitive sense organs.
    It